# Experiment 1 — Word-context representation

In [10]:
import pandas as pd
import re
file_path = '../lab01/c4-train.00000-of-01024-30K.json'
df = pd.read_json(file_path, lines=True)

df_sampled = df.sample(n=10000, random_state=42)

def tokenize(text):
  return re.findall(r'\b\w+\b', str(text).lower())

corpus = df_sampled['text'].apply(tokenize).tolist()
corpus = [doc for doc in corpus if len(doc) > 0]

In [11]:
from cooccurrence import MyCooccurrence

windows = [1, 2, 5]
for window in windows:
  print(f"\n{'='*40}")
  print(f"WINDOW SIZE = {window}")
  print(f"{'='*40}")

  model = MyCooccurrence()
  model.build_vocabulary(corpus, min_count=5)
  model.build_cooccurrence_matrix(corpus, window_size=window)
  matrix_shape = model.matrix.shape
  nnz = model.matrix.nnz
  total_elements = matrix_shape[0] * matrix_shape[1]
  sparsity = 100 - (nnz / total_elements * 100)
  print(f"Vocabulary Size:   {model.vocab_size} words")
  print(f"Matrix Size:       {matrix_shape[0]} x {matrix_shape[1]} cells")
  print(f"Non-zero values:   {nnz:,}")
  print(f"Matrix Sparsity:   {sparsity:.4f}%")

  try:
    doctor_vec = model.matrix[model.word2id["doctor"]].toarray().flatten()
    physician_vec = model.matrix[model.word2id["physician"]].toarray().flatten()
    
    sim = model.cosine_similarity(doctor_vec, physician_vec)
    print(f"Similarity (doctor - physician): {sim:.4f}")
  except KeyError:
    print("Similarity test failed: 'doctor' or 'physician' is not in this random 10k sample.")


WINDOW SIZE = 1
Vocabulary Size:   29381 words
Matrix Size:       29381 x 29381 cells
Non-zero values:   2,099,883
Matrix Sparsity:   99.7567%
Similarity (doctor - physician): 0.8355

WINDOW SIZE = 2
Vocabulary Size:   29381 words
Matrix Size:       29381 x 29381 cells
Non-zero values:   4,078,885
Matrix Sparsity:   99.5275%
Similarity (doctor - physician): 0.8487

WINDOW SIZE = 5
Vocabulary Size:   29381 words
Matrix Size:       29381 x 29381 cells
Non-zero values:   8,885,616
Matrix Sparsity:   98.9707%
Similarity (doctor - physician): 0.8940


In [12]:
test_words = ["doctor", "hospital", "car", "computer", "food"]

for word in test_words:
  print(f"Word: '{word}'")
  try:
    results = model.most_similar(word, top_k=5)
    for rank, (match_word, score) in enumerate(results, 1):
      print(f"  {rank}. {match_word} (score: {score:.4f})")
  except KeyError:
    print(f"'{word}' is not in the vocabulary.")

Word: 'doctor'
  1. child (score: 0.9456)
  2. vehicle (score: 0.9248)
  3. plan (score: 0.9242)
  4. or (score: 0.9220)
  5. home (score: 0.9209)
Word: 'hospital'
  1. house (score: 0.9404)
  2. public (score: 0.9403)
  3. community (score: 0.9373)
  4. office (score: 0.9360)
  5. school (score: 0.9347)
Word: 'car'
  1. while (score: 0.9744)
  2. house (score: 0.9720)
  3. company (score: 0.9693)
  4. after (score: 0.9671)
  5. game (score: 0.9667)
Word: 'computer'
  1. business (score: 0.9407)
  2. home (score: 0.9370)
  3. budget (score: 0.9349)
  4. account (score: 0.9333)
  5. journey (score: 0.9327)
Word: 'food'
  1. music (score: 0.9656)
  2. water (score: 0.9654)
  3. local (score: 0.9634)
  4. other (score: 0.9630)
  5. performance (score: 0.9629)


# Experiment 2 — Word2Vec

In [13]:
%%capture
from gensim.models import Word2Vec
w2v = Word2Vec(sentences=corpus, vector_size=100, window=5, min_count=5, epochs=10, seed=42,workers=4)

In [14]:
test_words = ["doctor", "hospital", "patient", "disease", "computer", "football", "banana"]
for word in test_words:
  print(f"Word: '{word}'")
  try:
    results = w2v.wv.most_similar(word, topn=5)
    for rank, (sim_word, score) in enumerate(results, 1):
      print(f"  {rank}. {sim_word} ({score:.4f})")
  except KeyError:
    print(f"'{word}' not in vocabulary.")

Word: 'doctor'
  1. physician (0.8012)
  2. spouse (0.6997)
  3. child (0.6612)
  4. novak (0.6528)
  5. nurse (0.6337)
Word: 'hospital'
  1. clinic (0.7202)
  2. alabama (0.6917)
  3. cincinnati (0.6657)
  4. pennsylvania (0.6626)
  5. attorneys (0.6335)
Word: 'patient'
  1. treatment (0.6902)
  2. physician (0.6855)
  3. patients (0.6491)
  4. doctor (0.6298)
  5. medication (0.6224)
Word: 'disease'
  1. chronic (0.8138)
  2. diabetes (0.8086)
  3. diseases (0.8003)
  4. infections (0.7996)
  5. syndrome (0.7996)
Word: 'computer'
  1. desktop (0.7820)
  2. browser (0.7604)
  3. laptop (0.7338)
  4. software (0.7179)
  5. device (0.7154)
Word: 'football'
  1. soccer (0.8100)
  2. baseball (0.8035)
  3. basketball (0.7718)
  4. tournaments (0.7529)
  5. tournament (0.7159)
Word: 'banana'
  1. peppers (0.9156)
  2. flavored (0.9083)
  3. olives (0.9060)
  4. thyme (0.9053)
  5. cabbage (0.9009)


# Experiment 3 — Context window

In [15]:
import pandas as pd
word_pairs = [
  ("doctor", "physician"),
  ("doctor", "hospital"),
  ("cat", "dog")
]
results_dict = {
  "Word pair": [f"{w1} - {w2}" for w1, w2 in word_pairs]
}
windows_to_test = [2, 5, 10]


In [16]:
for win in windows_to_test:
  print(f"Word2Vec (Window={win})")
  model_exp3 = Word2Vec(
    sentences=corpus,
    vector_size=100,      
    window=win,           
    min_count=5,
    epochs=10,
    sg=1,                 
    seed=42,
    workers=4
  )
  sim_scores = []
  for w1, w2 in word_pairs:
    try:
      score = model_exp3.wv.similarity(w1, w2)
      sim_scores.append(round(score, 4))
    except KeyError:
      sim_scores.append(None)
  results_dict[f"Window {win}"] = sim_scores


Word2Vec (Window=2)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec (Window=5)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

Word2Vec (Window=10)


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [17]:
df_exp3 = pd.DataFrame(results_dict)
display(df_exp3)

,Word pair,Window 2,Window 5,Window 10
0,doctor - physician,0.7527,0.7376,0.6929
1,doctor - hospital,0.4862,0.5552,0.5092
2,cat - dog,0.6559,0.6341,0.6591


# Experiment 4 — Embedding dimension

In [18]:
import time
import pandas as pd

dimensions = [50, 100, 300]

results_dim = {
  "Dimension": [],
  "Training Time (s)": [],
  "Vector Matrix Size": [],
  "Sim (doctor-physician)": [],
  "Analogy (king-man+woman)": []
}


for dim in dimensions:
  start_time = time.time()
  model_exp4 = Word2Vec(
    sentences=corpus,
    vector_size=dim,     
    window=5,             
    min_count=5,
    epochs=10,
    sg=1,                 
    seed=42,
    workers=4
  )
  train_time = round(time.time() - start_time, 2)
  vocab_size = len(model_exp4.wv.index_to_key)
  matrix_size = vocab_size * dim
  try:
    sim = model_exp4.wv.similarity("doctor", "physician")
    sim_score = round(sim, 4)
  except KeyError:
    sim_score = None
  try:
    analogy_result = model_exp4.wv.most_similar(positive=["king", "woman"], negative=["man"], topn=1)
    analogy_word = analogy_result[0][0]
  except KeyError:
    analogy_word = "N/A"
  results_dim["Dimension"].append(dim)
  results_dim["Training Time (s)"].append(train_time)
  results_dim["Vector Matrix Size"].append(matrix_size)
  results_dim["Sim (doctor-physician)"].append(sim_score)
  results_dim["Analogy (king-man+woman)"].append(analogy_word)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_fl

In [19]:
df_exp4 = pd.DataFrame(results_dim)
display(df_exp4)

,Dimension,Training Time (s),Vector Matrix Size,Sim (doctor-physician),Analogy (king-man+woman)
0,50,32.42,1469050,0.7996,queen
1,100,37.87,2938100,0.7024,queen
2,300,59.96,8814300,0.5999,queen


# Application — Semantic Search

In [20]:
import numpy as np
def get_document_embedding(tokens, w2v_model):
  vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv.key_to_index]
  if len(vectors) == 0:
    return np.zeros(w2v_model.vector_size)
  return np.mean(vectors, axis=0)
doc_embeddings = np.array([get_document_embedding(doc, model_exp4) for doc in corpus])

def semantic_search(query_str, w2v_model, doc_matrix, original_corpus, top_k=3):
  query_tokens = re.findall(r'\b\w+\b', query_str.lower())
  query_vec = get_document_embedding(query_tokens, w2v_model)
  query_norm = np.linalg.norm(query_vec)
  
  if query_norm == 0:
    return "Query words are not in the vocabulary"
  dot_products = doc_matrix.dot(query_vec)
  doc_norms = np.linalg.norm(doc_matrix, axis=1)
  similarities = dot_products / (doc_norms * query_norm + 1e-9)
  best_indices = np.argsort(similarities)[::-1][:top_k]
  print(f"SEARCH QUERY: '{query_str}'")
  for rank, idx in enumerate(best_indices, 1):
      score = similarities[idx]
      doc_text = " ".join(original_corpus[idx])
      print(f"\n[Rank {rank} | Score: {score:.4f}]")
      print(f"{doc_text}\n")
      

In [21]:
semantic_search("medical treatment", model_exp4, doc_embeddings, corpus, top_k=5)
semantic_search("delicious food recipe", model_exp4, doc_embeddings, corpus, top_k=5)
semantic_search("implement linear regression", model_exp4, doc_embeddings, corpus, top_k=5)

SEARCH QUERY: 'medical treatment'

[Rank 1 | Score: 0.6464]
a phase ii pilot study to evaluate use of intravenous lidocaine for opioid refractory pain in cancer patients kerala india status of cancer pain relief and palliative care nurse moral distress and cancer pain management an ethnography of oncology nurses in india


[Rank 2 | Score: 0.5969]
the indiana hospital heart institute has one of the largest most experienced cardiac surgery team in south india over the last 2 1 2 years more than 200 patients have had their heart surgery performed at indiana hospital heart institute for ranging from valve surgery repairs to cabg s complex aortic and arch surgeries the indiana hospital heart institute performs coronary artery bypass operations with a success rate of over 99 even complex operations re do surgeries and aortic and great vessel surgeries are routinely performed at indiana hospital heart institute with excellent results department of cardiothoracic and vascular surgeries offer 